## Baseline RF

In [0]:
import pandas as pd

train_pdf = pd.read_parquet("/Volumes/workspace/default/cancer_data/03_train.parquet")
test_pdf = pd.read_parquet("/Volumes/workspace/default/cancer_data/03_test.parquet")
print(f"Loaded {len(train_pdf)} train rows, {len(test_pdf)} test rows")

Loaded 58767 train rows, 14692 test rows


In [0]:
# __________________________
# OHE COLUMNS DEFINITION
# (Required to identify generated One-Hot Encoded features from previous notebook)
# __________________________
ohe_cols_def = [
    "raca_cor",
    "uf_procedencia_regiao",
    "uf_hospital_regiao",
    "origem_encaminhamento",
    "exames_relevantes_diagnostico",
    "diagnostico_tratamento_anteriores",
    "base_diagnostico_mais_importante",
    "base_diagnostico_microscopica",
    "primeiro_tratamento_hospital",
    "razao_nao_tratamento_hospital",
    "historico_tabagismo_clinico",
    "historico_alcoolismo_clinico",
]

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# __________________________
# FEATURES DO MODELO
# tempo_ate_obito excluído intencionalmente — é dado pós-desfecho
# no RF normal entra apenas como baseline de classificação binária (status_vital)
# a estrutura de censura será tratada adequadamente no RSF
# __________________________

feature_cols = [
    # numéricas reais
    "idade",
    "tempo_ate_consulta",
    "tempo_ate_tratamento",
    # binárias (já mapeadas para 0/1)
    "tipo_caso",
    "sexo",
    "historico_familiar_cancer",
    "mais_de_um_tumor_primario",
    # ordinais (label encoding — progressão clínica real)
    "escolaridade",
    "t_tnm",
    "n_tnm",
    "m_tnm",
    "t_ptnm",
    "n_ptnm",
    "m_ptnm",
    "comportamento_histologico_tumor",
    # flags de ausência dos hábitos (extraídos na recodificação especial)
    "historico_tabagismo_info_ausente",
    "historico_alcoolismo_info_ausente",
    # target encoding (alta cardinalidade)
    "tipo_histologico_tumor_te",
    "subcat_localizacao_primaria_te",
    "cat_localizacao_primaria_te",  # adicionado — movido do OHE
    # gap encoding
    "ocupacao_principal_gap",
]

# __________________________
# ACRESCENTA COLUNAS OHE
# get_feature_names_out gera nomes no formato {coluna_original}_{categoria}
# ex: raca_cor_Branca, uf_procedencia_regiao_Sudeste
# filtra apenas as que realmente existem no treino (segurança contra colunas ausentes)
# __________________________

feature_cols += [
    c for c in train_pdf.columns
    if any(c.startswith(f"{base}_") for base in ohe_cols_def)
]
feature_cols = [c for c in feature_cols if c in train_pdf.columns]

X_train = train_pdf[feature_cols].fillna(0)
y_train = train_pdf["status_vital"]
w_train = train_pdf["class_weight"]

X_test = test_pdf[feature_cols].fillna(0)
y_test = test_pdf["status_vital"]

# __________________________
# RANDOM FOREST — BASELINE
# Este modelo trata o problema como classificação binária simples
# Ignora a estrutura de censura: pacientes vivos são tratados como "não morreram ainda"
# e não como "morreram em data desconhecida após o follow-up"
# O AUC aqui serve apenas como referência para comparação com o RSF
# __________________________

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
)

rf.fit(X_train, y_train, sample_weight=w_train)

# predict_proba retorna [P(classe=0), P(classe=1)] — pegamos a coluna 1 (óbito=0, vivo=1)
# ATENÇÃO: se y_train tiver apenas uma classe, [:, 1] lança IndexError
# isso foi corrigido ao remover status_vital do map_bin (bug anterior)
y_prob = rf.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
print({"train_rows": len(train_pdf), "test_rows": len(test_pdf), "auc": round(auc, 4)})

{'train_rows': 58767, 'test_rows': 14692, 'auc': 0.8698}


## Baseline RSF

In [0]:
%pip install scikit-survival==0.22.2

In [0]:
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored, integrated_brier_score
from sksurv.util import Surv
import numpy as np

# __________________________
# VARIÁVEL DE DESFECHO PARA RSF
# sksurv exige um array estruturado com dtype=[('event', bool), ('time', float)]
# event = True  → óbito ocorreu    (status_vital = 0)
# event = False → censurado/vivo   (status_vital = 1)
#
# ATENÇÃO: inversão intencional — no dataset 0=Óbito, 1=Vivo
# no sksurv o evento de interesse é a morte, então invertemos para bool
#
# tempo: para quem morreu     → tempo_ate_obito
#        para quem foi censurado → tempo_total_doenca (último follow-up)
# __________________________

def build_survival_array(pdf):
    event = (pdf["status_vital"] == 0).astype(bool)  # True = morreu
    time  = np.where(
        event,
        pdf["tempo_ate_obito"],       # tempo até o óbito
        pdf["tempo_total_doenca"],    # tempo até censura (último follow-up)
    ).astype(float)
    return Surv.from_arrays(event=event, time=time)

y_train_surv = build_survival_array(train_pdf)
y_test_surv  = build_survival_array(test_pdf)

# __________________________
# FEATURES — mesmas do RF baseline, exceto que tempo_total_doenca
# não entra em X (ele compõe y_surv como variável de tempo)
# __________________________

# Usar as mesmas features do RF
feature_cols_rsf = feature_cols

X_train_rsf = train_pdf[feature_cols_rsf].astype(float)
X_test_rsf  = test_pdf[feature_cols_rsf].astype(float)

# __________________________
# RANDOM SURVIVAL FOREST
# Diferença fundamental em relação ao RF baseline:
# — aprende a função de sobrevivência S(t) em vez de classificar 0/1
# — trata censura corretamente: pacientes vivos contribuem com informação
#   até o momento da censura, sem assumir que sobreviverão para sempre
# — log_rank_score como critério de split: maximiza separação das curvas
#   de sobrevivência em cada nó (equivalente ao log-rank test de Kaplan-Meier)
# __________________________

rsf = RandomSurvivalForest(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,   # evita splits em nós muito pequenos
    min_samples_leaf=5,     # garante estimativas estáveis nas folhas
    n_jobs=-1,
    random_state=42,
)

rsf.fit(X_train_rsf, y_train_surv)

# __________________________
# MÉTRICAS DE AVALIAÇÃO
#
# 1) C-index (Harrell): proporção de pares concordantes
#    — análogo ao AUC para dados de sobrevivência
#    — 0.5 = aleatório, 1.0 = perfeito
#
# 2) Brier Score integrado (IPCW): erro quadrático médio das probabilidades
#    de sobrevivência ao longo do tempo
#    — penaliza calibração além de discriminação
#    — 0.0 = perfeito, 0.25 = modelo nulo (sem informação)
#    — calculado apenas no intervalo seguro: 10º–90º percentil dos tempos
#      para evitar instabilidade nas caudas onde há poucos eventos
# __________________________

# C-index
risk_scores = rsf.predict(X_test_rsf)  # score de risco: quanto maior, maior risco de óbito
event_test  = y_test_surv["event"]
time_test   = y_test_surv["time"]

cindex, concordant, discordant, tied_risk, tied_time = concordance_index_censored(
    event_test, time_test, risk_scores
)

# Brier Score integrado
# define intervalo seguro de avaliação
t_min = np.percentile(time_test[event_test], 10)
t_max = np.percentile(time_test[event_test], 90)
times_grid = np.linspace(t_min, t_max, 100)

surv_funcs   = rsf.predict_survival_function(X_test_rsf)
surv_matrix  = np.row_stack([fn(times_grid) for fn in surv_funcs])

ibs = integrated_brier_score(y_train_surv, y_test_surv, surv_matrix, times_grid)

print({
    "c_index":               round(cindex, 4),
    "integrated_brier_score": round(ibs, 4),
    "concordant_pairs":      concordant,
    "discordant_pairs":      discordant,
    "train_rows":            len(train_pdf),
    "test_rows":             len(test_pdf),
})

## Baseline RF Regressor

In [0]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# __________________________
# SUBSET: apenas pacientes que morreram (status_vital=0)
# Para esses, tempo_ate_obito é conhecido e não nulo
# Censurados são excluídos — prever tempo para quem ainda está vivo
# não faz sentido sem modelagem de censura (isso é papel do RSF)
# __________________________

train_reg = train_pdf[train_pdf["status_vital"] == 0].copy()
test_reg  = test_pdf[test_pdf["status_vital"] == 0].copy()

print({
    "train_obitos": len(train_reg),
    "test_obitos":  len(test_reg),
    "pct_train":    round(len(train_reg) / len(train_pdf) * 100, 1),
    "pct_test":     round(len(test_reg)  / len(test_pdf)  * 100, 1),
})

# __________________________
# FEATURES — mesmas do RSF, exceto tempo_total_doenca, tempo_ate_obito (é o target aqui)
# e exceto status_vital (constante nesse subset — todos são 0)
# __________________________

feature_cols_reg = [
    c for c in feature_cols_rsf
    if c not in ("tempo_ate_obito", "status_vital", "tempo_total_doenca") #NOTE isso ta meio redundante, já está sem essas colunas
]
feature_cols_reg = [c for c in feature_cols_reg if c in train_reg.columns]

X_train_reg = train_reg[feature_cols_reg].astype(float)
y_train_reg = train_reg["tempo_ate_obito"].astype(float)

X_test_reg  = test_reg[feature_cols_reg].astype(float)
y_test_reg  = test_reg["tempo_ate_obito"].astype(float)

# __________________________
# RANDOM FOREST REGRESSOR — BASELINE
# Prediz o tempo até óbito em dias
# Limitações importantes desse modelo:
# — treinado apenas em quem morreu: pode não generalizar para o perfil dos censurados
# — ignora completamente a informação dos pacientes vivos durante o treino
# — sem modelagem de censura: é um baseline ingênuo, não um modelo de sobrevivência
# O RSF supera essas limitações ao usar todos os pacientes corretamente
# __________________________

rfr = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)

rfr.fit(X_train_reg, y_train_reg)

y_pred = rfr.predict(X_test_reg)

# __________________________
# MÉTRICAS
# MAE:  erro médio absoluto em dias — mais interpretável clinicamente
# RMSE: penaliza erros grandes — sensível a outliers de tempo longo
# baseline ingênuo (prever sempre a média) incluído para referência
# __________________________

mae_score          = mean_absolute_error(y_test_reg, y_pred)
rmse_score         = np.sqrt(mean_squared_error(y_test_reg, y_pred))

baseline_mae_score  = mean_absolute_error(y_test_reg, np.full_like(y_test_reg, y_train_reg.mean()))
baseline_rmse_score = np.sqrt(mean_squared_error(y_test_reg, np.full_like(y_test_reg, y_train_reg.mean())))

print({
    "mae_dias":        round(mae_score, 1),
    "rmse_dias":       round(rmse_score, 1),
    "baseline_mae":    round(baseline_mae_score, 1),
    "baseline_rmse":   round(baseline_rmse_score, 1),
    "media_treino":    round(y_train_reg.mean(), 1),
    "mediana_treino":  round(y_train_reg.median(), 1),
})

## Diagnóstico de overfitting?

Utilizando os mesmos modelos e hiperparâmetros de antes.

**Validação cruzada (cross-validation)**  
É uma forma de testar se o modelo generaliza bem. Reduz o risco de confiar em um único split treino/teste. Útil para comparar modelos/hiperparâmetros.

Como funciona (ex.: 5 folds):
1. Divide os dados em 5 partes.
2. Treina em 4 partes e testa na parte restante.
3. Repete 5 vezes, trocando a parte de teste.
4. Faz a média dos scores.


**Permutation Importance**  
É uma forma de medir o quanto cada feature realmente ajuda na predição.

Como funciona:
1. Mede o score normal do modelo (ex.: AUC ou C-index).
2. Embaralha uma feature (uma de cada vez), quebrando sua relação com o alvo.
3. Mede o score de novo.
4. A queda no score = importância da feature.

In [0]:
import pandas as pd

train_pdf = pd.read_parquet("/Volumes/workspace/default/cancer_data/03_train.parquet")
test_pdf = pd.read_parquet("/Volumes/workspace/default/cancer_data/03_test.parquet")
print(f"Loaded {len(train_pdf)} train rows, {len(test_pdf)} test rows")

### RF with CV and permutation importance

In [0]:
# 2) Treina RF (Já treinado como `rf` acima)
# O modelo 'rf' já foi treinado (fit) no bloco Baseline RF.

rf_perm = permutation_importance(
    rf,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

rf_importances = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_perm.importances_mean,
    "std": rf_perm.importances_std
}).sort_values("importance", ascending=False)

display(rf_importances.head(10))

### RSF with CV and permutation importance

In [0]:
# 4) Prepara e treina RSF (Já treinado como `rsf` acima)
# O modelo 'rsf' já foi treinado (fit) no bloco Baseline RSF.

# 6) Permutation importance
rsf_perm = permutation_importance(
    rsf,
    X_test_rsf,
    y_test_surv,
    scoring=rsf_cindex_scorer,
    n_repeats=5, # Aumentado de 2 para 5 já que arrumamos a profundidade
    random_state=42,
    n_jobs=-1
)

rsf_importances = pd.DataFrame({
    "feature": feature_cols_rsf,
    "importance": rsf_perm.importances_mean,
    "std": rsf_perm.importances_std
}).sort_values("importance", ascending=False)

display(rsf_importances.head(10))

In [0]:
# so pra garantir
print("tempo_total_doenca" in X_test_rsf.columns)

In [0]:
from sklearn.model_selection import KFold

# 5) CV
rsf_cv = KFold(n_splits=5, shuffle=True, random_state=42)
rsf_cv_scores = cross_val_score(
    clone(rsf), # Clone copia max_depth=10, min_samples_leaf=5 evitando travamentos!
    X_train_rsf,
    y_train_surv,
    scoring=rsf_cindex_scorer,
    cv=rsf_cv,
    n_jobs=-1,
)

print(f"[RSF] C-index CV Mean : {np.mean(rsf_cv_scores):.4f}")
print(f"[RSF] C-index CV Std  : {np.std(rsf_cv_scores):.4f}")

In [0]:
# 6) Permutation importance
rsf_perm = permutation_importance(
    rsf_local,
    X_test_rsf,
    y_test_surv,
    scoring=rsf_cindex_scorer,
    n_repeats=2,
    random_state=42,
    n_jobs=-1,
)
rsf_imp_df = (
    pd.DataFrame({
        "feature": X_test_rsf.columns,
        "importance_mean": rsf_perm.importances_mean,
        "importance_std": rsf_perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)

print("\nTop 15 RSF (Permutation Importance):")
display(rsf_imp_df.head(15))

plt.figure(figsize=(8, 5))
rsf_top = rsf_imp_df.head(15).iloc[::-1]
plt.barh(rsf_top["feature"], rsf_top["importance_mean"])
plt.title("RSF - Top 15 Permutation Importance (teste)")
plt.xlabel("Importância média")
plt.tight_layout()
plt.show()

In [0]:
#TODO: fazer busca por melhores hiperparametros grid search cv